In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from google.colab import files, drive

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras import layers, models, optimizers, callbacks, regularizers

plt.rcParams['figure.figsize'] = (8, 6)


print("=== MODULE 1: FOUNDATIONS OF VISION ===")
print("Upload ONE image for Module 1 (any leaf / any picture)...")

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

bgr = cv2.imread(file_name)
if bgr is None:
    raise ValueError("Invalid image uploaded")

original = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
original = cv2.resize(original, (256, 256))

plt.imshow(original)
plt.title("Original Image (Module 1)")
plt.axis("off")
plt.show()

def show_side_by_side(img1, title1, img2, title2, gray=False):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    if gray:
        ax[0].imshow(img1, cmap='gray')
        ax[1].imshow(img2, cmap='gray')
    else:
        ax[0].imshow(img1)
        ax[1].imshow(img2)
    ax[0].set_title(title1)
    ax[1].set_title(title2)
    ax[0].axis("off")
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()

# geometric transforms
angle = float(input("Enter rotation angle (e.g. 30, -45, 90): "))

h, w = original.shape[:2]
center = (w // 2, h // 2)

rotated = cv2.warpAffine(original, cv2.getRotationMatrix2D(center, angle, 1), (w, h))
flipped = cv2.flip(original, 1)
scaled = cv2.resize(original, None, fx=1.5, fy=1.5)

show_side_by_side(original, "Original", rotated, f"Rotated {angle} deg")
show_side_by_side(original, "Original", flipped, "Horizontal Flip")
show_side_by_side(original, "Original", scaled, "Scaled 1.5x")

# intensity transforms
alpha, beta = 1.3, 30
bright_contrast = cv2.convertScaleAbs(original, alpha=alpha, beta=beta)

gamma = 0.6
table = np.array([(i / 255) ** (1 / gamma) * 255 for i in range(256)]).astype("uint8")
gamma_corrected = cv2.LUT(original, table)

gray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
equalized = cv2.equalizeHist(gray)

show_side_by_side(original, "Original", bright_contrast, "Brightness / Contrast")
show_side_by_side(original, "Original", gamma_corrected, "Gamma Correction")
show_side_by_side(gray, "Gray", equalized, "Histogram Equalized", gray=True)

plt.figure()
plt.hist(gray.flatten(), 256)
plt.title("Original Histogram")
plt.show()

plt.figure()
plt.hist(equalized.flatten(), 256)
plt.title("Equalized Histogram")
plt.show()

# noise models
def gaussian_noise(img, std=25):
    noise = np.random.normal(0, std, img.shape)
    noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return noisy

def salt_pepper(img, amount=0.03):
    noisy = img.copy()
    h, w = img.shape[:2]
    num = int(amount * h * w)

    y = np.random.randint(0, h, num)
    x = np.random.randint(0, w, num)
    noisy[y, x] = [255, 255, 255]

    y = np.random.randint(0, h, num)
    x = np.random.randint(0, w, num)
    noisy[y, x] = [0, 0, 0]

    return noisy

gauss_noisy = gaussian_noise(original)
sp_noisy = salt_pepper(original)

show_side_by_side(original, "Original", gauss_noisy, "Gaussian Noise")
show_side_by_side(original, "Original", sp_noisy, "Salt & Pepper Noise")

# restoration filters
gauss_blur = cv2.GaussianBlur(gauss_noisy, (5, 5), 1.2)
bilateral = cv2.bilateralFilter(gauss_noisy, 9, 75, 75)
median_filtered = cv2.medianBlur(sp_noisy, 5)

show_side_by_side(gauss_noisy, "Gaussian Noisy", gauss_blur, "Gaussian Blur")
show_side_by_side(gauss_noisy, "Gaussian Noisy", bilateral, "Bilateral Filter")
show_side_by_side(sp_noisy, "Salt & Pepper", median_filtered, "Median Filter")

# edge detection
def edges(img):
    return cv2.Canny(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), 100, 200)

e_org = edges(original)
e_gn = edges(gauss_noisy)
e_gb = edges(gauss_blur)
e_bi = edges(bilateral)
e_sp = edges(sp_noisy)
e_med = edges(median_filtered)

def show_edges_row(images, titles):
    fig, ax = plt.subplots(1, len(images), figsize=(18, 4))
    for a, im, t in zip(ax, images, titles):
        a.imshow(im, cmap='gray')
        a.set_title(t)
        a.axis("off")
    plt.tight_layout()
    plt.show()

show_edges_row(
    [e_org, e_gn, e_gb, e_bi],
    ["Original", "Gaussian Noise", "Gaussian Blur", "Bilateral Filter"]
)
show_edges_row(
    [e_org, e_sp, e_med],
    ["Original", "Salt & Pepper", "Median Filter"]
)

# edge pixel count
def count_edges(e):
    return np.sum(e > 0)

counts = {
    "Original": count_edges(e_org),
    "Gauss Noise": count_edges(e_gn),
    "Gauss Blur": count_edges(e_gb),
    "Bilateral": count_edges(e_bi),
    "Salt & Pepper": count_edges(e_sp),
    "Median": count_edges(e_med),
}

print("\nEDGE PIXELS COUNT:")
for k, v in counts.items():
    print(f"{k:15}: {v}")

plt.figure()
plt.bar(counts.keys(), counts.values())
plt.xticks(rotation=20)
plt.title("Edge Pixel Comparison")
plt.show()

print("\n=== MODULE 1 COMPLETE ===")

# MODULE 2

print("\n\n=== MODULE 2: CLASSICAL FEATURE-BASED VISION ===")
print("Upload MULTIPLE images (different leaves / classes)...")

uploaded = files.upload()
image_names = list(uploaded.keys())

if len(image_names) < 2:
    raise ValueError("Please upload at least 2 images for Module 2")

rgb_first = cv2.cvtColor(cv2.imread(image_names[0]), cv2.COLOR_BGR2RGB)
rgb_first = cv2.resize(rgb_first, (256, 256))
gray_first = cv2.cvtColor(rgb_first, cv2.COLOR_RGB2GRAY)

plt.imshow(rgb_first)
plt.title("Dataset Image 1 (Module 2)")
plt.axis("off")
plt.show()

# feature functions
def feat_hog(g):
    f, _ = hog(
        g, orientations=9, pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), visualize=True, block_norm='L2-Hys'
    )
    return f

def feat_lbp_hist(g):
    l = local_binary_pattern(g, 8, 1, method="uniform")
    h, _ = np.histogram(l.ravel(), bins=np.arange(0, 10), density=True)
    return h

def feat_glcm(g):
    s = (g / 16).astype(np.uint8)
    gl = graycomatrix(s, [1], [0], levels=16, symmetric=True, normed=True)
    return np.array([
        graycoprops(gl, 'contrast')[0, 0],
        graycoprops(gl, 'homogeneity')[0, 0],
        graycoprops(gl, 'energy')[0, 0],
        graycoprops(gl, 'correlation')[0, 0]
    ])

def feat_hu(g):
    m = cv2.moments(g)
    hu = cv2.HuMoments(m)
    return (-np.sign(hu) * np.log10(np.abs(hu) + 1e-10)).flatten()

print("\nShowing keypoints and descriptors on first image...\n")

# SIFT keypoints
sift_vis_tool = cv2.SIFT_create()
sift_kp, _ = sift_vis_tool.detectAndCompute(gray_first, None)
sift_vis = cv2.drawKeypoints(
    rgb_first, sift_kp, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
plt.imshow(sift_vis)
plt.title("SIFT Keypoints")
plt.axis("off")
plt.show()

# FAST keypoints
fast_vis_tool = cv2.FastFeatureDetector_create(threshold=25)
fast_kp = fast_vis_tool.detect(gray_first, None)
fast_vis = cv2.drawKeypoints(
    rgb_first, fast_kp, None,
    color=(255, 0, 0)
)
plt.imshow(fast_vis)
plt.title("FAST Keypoints")
plt.axis("off")
plt.show()

# hybrid SIFT and FAST
hybrid_img = rgb_first.copy()
hybrid_img = cv2.drawKeypoints(hybrid_img, sift_kp, None, color=(0, 255, 0))
hybrid_img = cv2.drawKeypoints(hybrid_img, fast_kp, None, color=(255, 0, 0))
plt.imshow(hybrid_img)
plt.title("Hybrid (SIFT + FAST)")
plt.axis("off")
plt.show()

# HOG visualization
hog_vec_first, hog_vis = hog(
    gray_first, orientations=9, pixels_per_cell=(8, 8),
    cells_per_block=(2, 2), visualize=True, block_norm='L2-Hys'
)
hog_norm = (hog_vis - hog_vis.min()) / (hog_vis.max() - hog_vis.min())
plt.imshow(hog_norm, cmap='gray')
plt.title("HOG Visualization")
plt.axis("off")
plt.show()

# LBP visualization
lbp_first = local_binary_pattern(gray_first, 8, 1, method="uniform")
plt.imshow(lbp_first, cmap='gray')
plt.title("LBP Texture Image")
plt.axis("off")
plt.show()

# GLCM bar plot
glcm_vals = feat_glcm(gray_first)
plt.figure()
plt.bar(["Contrast", "Homog.", "Energy", "Corr."], glcm_vals)
plt.title("GLCM Feature Values (First Image)")
plt.ylabel("Value")
plt.show()

# Hu moments bar plot
hu_vals = feat_hu(gray_first)
plt.figure()
plt.bar([f"Hu{i+1}" for i in range(7)], hu_vals)
plt.title("Hu Moments (log-scaled) - First Image")
plt.xticks(rotation=30)
plt.ylabel("Value")
plt.show()

print("\nKeypoint and descriptor visualizations complete.\n")

# feature extraction for all images
lbp_feats = []
fused_feats = []
all_sift_desc = []
desc_count = []

sift_tool = cv2.SIFT_create()

for name in image_names:
    rgb = cv2.cvtColor(cv2.imread(name), cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (128, 128))
    gray_img = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    h_feat = feat_hog(gray_img)
    l_feat = feat_lbp_hist(gray_img)
    g_feat = feat_glcm(gray_img)
    u_feat = feat_hu(gray_img)

    lbp_feats.append(l_feat)
    fused_feats.append(np.concatenate([h_feat, l_feat, g_feat, u_feat]))

    kp, desc = sift_tool.detectAndCompute(gray_img, None)
    if desc is not None:
        all_sift_desc.append(desc)
        desc_count.append(len(desc))
    else:
        desc_count.append(0)

lbp_feats = np.array(lbp_feats)
fused_feats = np.array(fused_feats)

print("LBP feature matrix shape   :", lbp_feats.shape)
print("Fused feature matrix shape :", fused_feats.shape)

# PCA and KMeans clustering
pca = PCA(n_components=2)
reduced = pca.fit_transform(fused_feats)

print("PCA variance ratio (2D):", pca.explained_variance_ratio_)

k = int(input("Enter KMeans cluster count (e.g. 2 or 3): "))

if k > len(reduced):
    print(f"Requested k={k} is > number of samples={len(reduced)}. Using k={len(reduced)}.")
    k = len(reduced)

km = KMeans(n_clusters=k, random_state=0)
labels = km.fit_predict(reduced)

plt.figure()
plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap='tab10')
plt.title("Clusters in PCA Space (Fused Features)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Cluster ID")
plt.show()

# Bag of visual words using SIFT
print("\nBuilding BoVW vocabulary using SIFT descriptors...")

if len(all_sift_desc) > 0:
    all_desc = np.vstack(all_sift_desc)
    km_vocab = KMeans(n_clusters=20, n_init=10, random_state=0)
    km_vocab.fit(all_desc)

    bovw = []
    ptr = 0
    for i, c in enumerate(desc_count):
        if c > 0:
            d = all_desc[ptr:ptr + c]
            ptr += c
            words = km_vocab.predict(d)
            hist, _ = np.histogram(words, bins=np.arange(0, 21), density=True)
        else:
            hist = np.zeros(20)
        bovw.append(hist)

    bovw = np.array(bovw)
    print("BoVW feature matrix shape:", bovw.shape)
else:
    bovw = None
    print("No SIFT descriptors found. BoVW not computed.")

# template matching baseline
print("\nTemplate Matching baseline between first two images:")

imgA = cv2.imread(image_names[0], 0)
imgB = cv2.imread(image_names[1], 0)

imgA = cv2.resize(imgA, (128, 128))
imgB = cv2.resize(imgB, (128, 128))

res = cv2.matchTemplate(imgB, imgA, cv2.TM_CCOEFF_NORMED)
score = np.max(res)

print("Similarity Score (Image 0 vs Image 1):", score)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(imgA, cmap='gray')
plt.title("Template (Image 0)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(imgB, cmap='gray')
plt.title("Search Image (Image 1)")
plt.axis("off")
plt.tight_layout()
plt.show()

print("\n=== MODULE 2 COMPLETE ===")

#MODULE3 DEEP LEARNING WITH MOBILENETV2

print("\n\n=== MODULE 3: DEEP LEARNING ON PLANT DISEASE DATASET ===")

#mount drive and extract dataset
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation.zip"
data_dir = "/content/Plant_leave_diseases_dataset_with_augmentation"

if not os.path.isdir(data_dir):
    import zipfile
    print("Unzipping dataset (one-time only)...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall("/content")
else:
    print("Dataset folder already exists, skipping unzip.")

print("Dataset root folder:", data_dir)
print("Example class folders:", sorted(os.listdir(data_dir))[:5])

# data generators
img_size = (128, 128)
batch = 16

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=(0.8, 1.2),
    validation_split=0.2
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    data_dir, target_size=img_size, batch_size=batch,
    class_mode="categorical", subset="training", shuffle=True
)

val_data = val_gen.flow_from_directory(
    data_dir, target_size=img_size, batch_size=batch,
    class_mode="categorical", subset="validation", shuffle=False
)

num_classes = train_data.num_classes
class_map = train_data.class_indices
idx_to_class = {v: k for k, v in class_map.items()}

print("\nNumber of classes:", num_classes)
print("Sample class mapping:", list(class_map.items())[:5])

# visualize augmented images
aug_images, aug_labels = next(iter(train_data))

def undo_preprocess(img_batch):
    img = (img_batch + 1.0) / 2.0
    img = np.clip(img, 0, 1)
    return img

aug_images_vis = undo_preprocess(aug_images)

plt.figure(figsize=(10, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(aug_images_vis[i])
    class_idx = np.argmax(aug_labels[i])
    plt.title(idx_to_class[class_idx][:12], fontsize=8)
    plt.axis("off")
plt.suptitle("Examples of Augmented Training Images", fontsize=12)
plt.tight_layout()
plt.show()

# build transfer learning model
base_model = MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=img_size + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(
    num_classes,
    activation="softmax",
    kernel_regularizer=regularizers.l2(1e-4)
)(x)

model = models.Model(inputs, outputs, name="PlantDisease_MobileNetV2")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# train classifier head only
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_phase1 = model.fit(
    train_data,
    epochs=5,
    validation_data=val_data,
    callbacks=[early_stop]
)

def plot_history(history, prefix=""):
    plt.figure()
    plt.plot(history.history["accuracy"], label="train_acc")
    plt.plot(history.history["val_accuracy"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(prefix + " Accuracy")
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(prefix + " Loss")
    plt.legend()
    plt.show()

plot_history(history_phase1, "Phase 1 (Frozen Base)")

val_loss, val_acc = model.evaluate(val_data, verbose=0)
print(f"\nValidation accuracy after Phase 1: {val_acc:.4f}")

# fine-tune top layers
fine_tune_at = len(base_model.layers) - 20

for i, layer in enumerate(base_model.layers):
    layer.trainable = (i >= fine_tune_at)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nFine-tuning last 20 layers of the base model...")
history_phase2 = model.fit(
    train_data,
    epochs=5,
    validation_data=val_data,
    callbacks=[early_stop]
)

plot_history(history_phase2, "Phase 2 (Fine-tuning)")

val_loss, val_acc = model.evaluate(val_data, verbose=0)
print(f"\nFINAL validation accuracy after fine-tuning: {val_acc:.4f}")

#MODULE3 SALIENCY MAP

def compute_saliency(prep_image_batch):
    img_tensor = tf.convert_to_tensor(prep_image_batch)
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        preds = model(img_tensor)
        top_class = tf.argmax(preds[0])
        top_score = preds[:, top_class]
    grads = tape.gradient(top_score, img_tensor)[0]
    sal = tf.reduce_max(tf.abs(grads), axis=-1)
    sal = sal.numpy()
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-10)
    return sal

val_images, val_labels = next(iter(val_data))
sample_prep = val_images[0]
true_label_idx = np.argmax(val_labels[0])
true_label_name = idx_to_class[true_label_idx]

sample_rel_path = val_data.filenames[0]
sample_full_path = os.path.join(data_dir, sample_rel_path)
orig_bgr = cv2.imread(sample_full_path)
orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
orig_rgb = cv2.resize(orig_rgb, img_size)

saliency = compute_saliency(sample_prep[np.newaxis, ...])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(orig_rgb)
plt.title(f"Original ({true_label_name})")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(saliency, cmap="hot")
plt.title("Saliency Map")
plt.axis("off")
plt.tight_layout()
plt.show()

#MODULE3 SIMPLE HSV SEGMENTATION

hsv = cv2.cvtColor(orig_rgb, cv2.COLOR_RGB2HSV)

lower_green = np.array([25, 40, 40])
upper_green = np.array([100, 255, 255])

mask = cv2.inRange(hsv, lower_green, upper_green)
segmented = cv2.bitwise_and(orig_rgb, orig_rgb, mask=mask)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(orig_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap="gray")
plt.title("Leaf Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(segmented)
plt.title("Segmented Leaf")
plt.axis("off")

plt.tight_layout()
plt.show()

print("\n=== MODULE 3 COMPLETE ===")

